# Prediction of overall survival patients with Myeloid Leukemia

## Importing Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Importing Data

### Clinical data

In [ ]:
X_clinic = pd.read_csv('../data/X_train/clinical_train.csv')
X_clinic = X_clinic.set_index('ID')
X_clinic

In [ ]:
X_clinic.dtypes

In [ ]:
X_clinic.describe()

In [ ]:
print(X_clinic['CENTER'].value_counts())
print(f'Cytogenetics cardinality: {X_clinic['CYTOGENETICS'].nunique()}')       # Karyotype cardinality
print(X_clinic['CYTOGENETICS'].value_counts().head(10))

### Molecular data

In [ ]:
X_molecular = pd.read_csv('../data/X_train/molecular_train.csv')
X_molecular = X_molecular.set_index('ID')
X_molecular

In [ ]:
X_molecular.dtypes

In [ ]:
X_molecular.describe()

### Labels

In [ ]:
Y = pd.read_csv('../data/target_train.csv')
Y = Y.set_index('ID')
Y

In [ ]:
Y.dtypes

In [ ]:
Y.describe()

### ID coherence

In [ ]:
unique_id = X_clinic.index.is_unique, X_molecular.index.is_unique, Y.index.is_unique
print(f'The ID is unique: X_clinic=  {unique_id[0]}, X_molecular= {unique_id[1]}, Y= {unique_id[2]}')
print(f'Number of non matching ID between X_clinic and Y: {len(Y.index.difference(X_clinic.index))}')
print(f'Number of non matching ID between X_molecular and Y: {len(Y.index.difference(X_molecular.index))}')

ID coherence in the datasets:
- In the clinical and labels datasets, IDs are unique.
- In the molecular dataset, IDs are not unique, a patient can have several rows.
- All patients in the labels dataset have an instance in the clinical datasets.
- Some patients don't have an instance in the molecular datasets.\
\
**A patient can have none, one or multiple mutations.**



## Labels: OS_YEARS & OS_STATUS

### Missing data in the labels

In [ ]:
print(f'Number of missing data: {Y.isna().sum()}')
print(f'Proportion of missing data: {Y.isna().sum()/Y.shape[0]}')
print(pd.crosstab(Y['OS_YEARS'].isna(), Y['OS_STATUS'].isna()))

150 data are missing in both OS_YEARS and OS_STATUS. \
Every missing data are missing from both labels. \
Missing data represents 4.5% of the dataset, we are removing them. \
We are storing the new labels in Yv.

In [ ]:
Yv = Y.dropna()
Yv

### Values and coherence between OS_YEARS and OS_STATUS

1. OS_STATUS takes 0 or 1
2. OS_YEARS should be positive
3. Patients with no follow-up or who were included at the very end of the study: OS_YEARS = 0 and OS_STATUS = 0
4. Distribution of survivability duration per status

In [ ]:
# 1. OS_STATUS takes 0 or 1
print(Yv['OS_STATUS'].value_counts(dropna=False))

In [ ]:
# 2. OS_STATUS >= 0
print(Yv['OS_YEARS'].describe())
print((Yv['OS_YEARS'] < 0).sum())

In [ ]:
# 3. Patient not under follow-up
print(Yv[Yv['OS_YEARS'] == 0]['OS_STATUS'].value_counts())

In [ ]:
# 4.
print(Yv.groupby('OS_STATUS')['OS_YEARS'].describe())
sns.histplot(data=Yv, x='OS_YEARS', hue='OS_STATUS', bins=44)

### Censorship

The data are right-censored. \
OS_YEARS represents the duration of the follow-up. A patient can be included in the study within the all duration of the study ~22 years. OS_STATUS indicates if the patient has survived at the end of the follow-up. However, we do not know if the follow-up has ended because the study has ended or for another reason (eg: the follow-up is done somewhere else, ...).

1. We are studying the censorship rate.
2. We are studying the distribution of the censorship in the data.

In [ ]:
# 1. Censorship rate
censoring_rate = (Yv['OS_STATUS'] == 0).mean()
print(f'Censorship rate: {censoring_rate:.1%}')

In [ ]:
# 2. Distribution of the censorship
censoring_distribution = Yv
censoring_distribution['bin'] = pd.cut(censoring_distribution['OS_YEARS'], bins = [0, 0.5, 1, 1.5, 2, 2.5, 3, 5, 7, 10, 25])
summary = censoring_distribution.groupby('bin', observed=True)['OS_STATUS'].agg(n='count', censoring_rate=lambda s: (s == 0).mean().round(2)*100)
summary

In [ ]:
sns.histplot(Yv, x=Yv['OS_YEARS'], hue=Yv['OS_STATUS'], bins=22, multiple='stack')

We do not have the date of inclusion, therefore it is hard to tell if the censorship is due to the end of the study or if the patients did not continue to participate in the study. \
- Censorship rate: 49,6%.
- Censorship distribution:
  - 49% before 6 month
  - then, a regular rising between 6 month and 7 years (67 % at 7 years)

The censorship distribution reveal a high proportion of censorship occurring within the first 6 months, which probably means that these were patients who, for one reason or another (short follow-up care, change of mind, ...), did not continue to participate in the study. \
The distribution for the rest of the group suggest that the censorship correspond to a patient alive at the end of the study.

### Index coherence

In [ ]:
print(Y.index.is_unique, X_clinic.index.is_unique)
print(Y.index.difference(X_clinic.index))
print(Y.index.is_unique, X_molecular.index.is_unique)
print(Y.index.difference(X_molecular.index))

### Data alignment

The data are not aligned between the 3 datasets. \
We are merging them on the patient ID in order to align it.\
The molecular set has one row per mutation per patient, so every patient has several rows.\
We are creating 2 dataset:
   - One with only the clinical data and the labels: clinical_dataset
   - One with all the data (clinical, molecular and labels: complete_dataset)

#### Clinical dataset

In [ ]:
clinical_dataset = X_clinic.merge(Y, how='inner', on='ID')
clinical_dataset

#### Complete dataset

In [ ]:
complete_dataset = X_clinic.merge(X_molecular, how='inner', on='ID')
complete_dataset = complete_dataset.merge(Y, how='inner', on='ID')
complete_dataset
print(np.unique_counts(complete_dataset.index))

## First visualisation

## Labels

In [ ]:
visualisation_label_dataset = Y
plt.figure(facecolor='w', figsize=(20, 10))

for index, column in enumerate(visualisation_label_dataset.columns):
    plt.subplot(1, 2, index+1)
    plt.hist(visualisation_label_dataset[column], color='blue')
    plt.title(column)
    plt.title(column)

### Clinical features

In [ ]:
visualisation_clinical_dataset = clinical_dataset.drop(columns=['CYTOGENETICS', 'OS_YEARS', 'OS_STATUS'])
plt.figure(facecolor='w', figsize=(20, 10))

for index, column in enumerate(visualisation_clinical_dataset.columns):
    plt.subplot(2, (len(visualisation_clinical_dataset.columns)+1)//2, index+1)
    plt.hist(visualisation_clinical_dataset[column], color='blue')
    plt.title(column)
    plt.title(column)


## Missing data in the features

### Clinical data

In [ ]:
clinical_dataset.isna().sum()

In [ ]:
print(clinical_dataset['WBC'].median())
print(clinical_dataset['WBC'].min())
print(clinical_dataset['WBC'].max())